In [1]:
from common_setup import *

In [2]:
import numpy as np
from scipy.optimize import minimize
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import ShuffleSplit
from typing import Tuple, List

def make_polynomial_features(X: np.ndarray, order: int) -> np.ndarray:
    """
    Generate polynomial features up to 'order'.
    If order=1, we get [1, x1, x2, ...].
    If order=2, we get [1, x1, x2, ..., x1^2, x1*x2, ...].
    """
    if order < 1 or order > 2:
        raise ValueError("Only order=1 or order=2 are supported.")
    
    poly = PolynomialFeatures(degree=order, include_bias=True)
    # shape: (n_samples, number_of_poly_features)
    return poly.fit_transform(X)


def sigmoid(s1,s2):
    diff=s1-s2
    ratio_pred=1.0/(1.0+np.exp(-diff))-0.5
    return ratio_pred

def inv_sigmoid(ratio_pred):
    return np.log((ratio_pred + 0.5) / (0.5 - ratio_pred))

def fit_definite_strength_poly(
    X1: np.ndarray,
    X2: np.ndarray,
    y: np.ndarray,
    order: int = 1,
    alpha: List[float] = None,
    w_init: np.ndarray = None
) -> np.ndarray:
    if alpha is None:
        # Default: no regularization if not specified
        alpha = [0.0] * (2 if order == 1 else 3)
    
    if order == 1 and len(alpha) != 2:
        raise ValueError("For order=1, alpha must have length=2.")
    if order == 2 and len(alpha) != 3:
        raise ValueError("For order=2, alpha must have length=3.")
    
    # Transform X1, X2 to polynomial features
    Z1 = make_polynomial_features(X1, order)  # shape: (n_samples, n_poly_features)
    Z2 = make_polynomial_features(X2, order)  # same shape
    n_samples, n_poly_features = Z1.shape
    
    d = X1.shape[1]
    intercept_idx = [0]
    linear_idx = list(range(1, d+1))
    if order == 1:
        quadratic_idx = []
    else:
        quadratic_idx = list(range(d+1, n_poly_features))
    
    alpha0 = alpha[0]  # intercept
    alpha1 = alpha[1]  # linear
    alpha2 = alpha[2] if order == 2 else 0.0  # quadratic (only if order=2)

    # Initial guess
    if w_init is None:
        w_init = np.zeros(n_poly_features)#np.random.randn(n_poly_features)

    def objective(w):
        # Predictions for each sample pair
        s1 = Z1.dot(w)  # shape: (n_samples,)
        s2 = Z2.dot(w)

        ratio_pred = sigmoid(s1,s2)  # shape: (n_samples,)
        # sum of squared errors
        residuals = y - ratio_pred
        sse = np.sum(residuals**2)

        # Regularization
        reg_intercept = alpha0 * np.sum(np.abs(w[intercept_idx]))
        reg_linear = alpha1 * np.sum(np.abs(w[linear_idx]))
        reg_quad = alpha2 * np.sum(np.abs(w[q]) for q in quadratic_idx)
        
        reg = reg_intercept + reg_linear + reg_quad
        return sse + reg

    # Minimize
    result = minimize(objective, w_init, method='BFGS', options={'maxiter': 100000} )
    w_opt = result.x
    return w_opt



def predict_ratio_poly(w: np.ndarray, X1: np.ndarray, X2: np.ndarray, order: int) -> np.ndarray:
    """
    Given fitted parameters w for polynomial of 'order',
    predict ratio = s1 / (s1 + s2).
    """
    Z1 = make_polynomial_features(X1, order)
    Z2 = make_polynomial_features(X2, order)
    s1 = Z1.dot(w)
    s2 = Z2.dot(w)
    return sigmoid(s1,s2)  #



In [3]:
import json
from pathlib import Path




In [4]:
def extract_sc_cc_numeric(loaded_results, rep_num, intensity_index):
    """
    Extracts and converts sc_list and cc_list for a specified intensity and replicate number.

    Parameters
    ----------
    loaded_results : dict
        The nested dictionary containing all results.
    rep_num : int
        The replicate number corresponding to 'community_n' (0 to 9).
    intensity_index : int, optional
        The index of the intensity to select (0-based). Defaults to 2 (third intensity).

    Returns
    -------
    sc_list_numeric : dict
        Sub-community list with integer keys.
    cc_list_numeric : dict
        Coalescence list with tuple integer keys.

    Raises
    ------
    ValueError
        If the loaded_results does not contain enough intensity levels,
        or if the specified community does not exist.
    """
    import sys

    # 1. Retrieve the specified intensity key
    intensity_keys = sorted(loaded_results.keys())
    if len(intensity_keys) <= intensity_index:
        raise ValueError(f"The loaded_results dictionary does not contain an intensity at index {intensity_index}.")

    intensity_key = intensity_keys[intensity_index]
    print(f"Selected Intensity: {intensity_key}")

    # 2. Construct the community key
    community_key = f'community_{rep_num}'
    print(f"Selected Community: {community_key}")

    # 3. Access the community data
    community_data = loaded_results.get(intensity_key, {}).get(community_key, {})
    if not community_data:
        raise ValueError(f"Community '{community_key}' not found under intensity '{intensity_key}'.")

    # 4. Extract sc_list and cc_list
    sc_list = community_data.get('sc_list', {})
    cc_list = community_data.get('cc_list', {})

    # 5. Convert sc_list keys from strings to integers
    sc_list_numeric = {}
    for k, v in sc_list.items():
        try:
            key_num = int(k)
            sc_list_numeric[key_num] = v
        except ValueError:
            print(f"Warning: Invalid sc_list key '{k}'. Skipping.", file=sys.stderr)

    # 6. Convert cc_list keys from 'x_y' strings to (x, y) tuples of integers
    cc_list_numeric = {}
    for k, v in cc_list.items():
        try:
            key_tuple = tuple(int(part) for part in k.split('_'))
            if len(key_tuple) != 2:
                raise ValueError
            cc_list_numeric[key_tuple] = v
        except ValueError:
            print(f"Warning: Invalid cc_list key '{k}'. Expected format 'x_y'. Skipping.", file=sys.stderr)

    return sc_list_numeric, cc_list_numeric


def prepare_X1_X2_y(loaded_results, rep_num, intensity_index,  eps=1e-3):
    # Extract sc_list and cc_list with numeric keys
    sc_list_numeric, cc_list_numeric = extract_sc_cc_numeric(loaded_results, rep_num, intensity_index)

    X1_all, X2_all, y_all = [], [], []

    # Iterate over each coalescence pair
    for (c1_idx, c2_idx), cmix in cc_list_numeric.items():
        c1 = sc_list_numeric.get(c1_idx)
        c2 = sc_list_numeric.get(c2_idx)

        if c1 is None or c2 is None:
            print(f"Warning: Sub-community indices {c1_idx}, {c2_idx} not found. Skipping.", file=sys.stderr)
            continue

        # Apply the metric function
        try:
            u, v, k = metric_VectorDecomposition_onlyPositive(c1, c2, cmix)
        except np.linalg.LinAlgError as e:
            if 'Singular matrix' in str(e):
                print("Singular matrix error occurred.")
                print(f"c1: {c1}")
                print(f"c2: {c2}")
                print(f"cmix: {cmix}")
                u1=u2=1/sqrt(2)
                k=None
                
        # Calculate y
        y = (np.arctan((np.abs(u) + eps) / (np.abs(v) + eps)) / (np.pi / 2)) - 0.5

        # Append to the lists
        X1_all.append(c1)
        X2_all.append(c2)
        y_all.append(y)

    return np.array(X1_all), np.array(X2_all), np.array(y_all)



def prepare_dominance(loaded_results, rep_num, intensity_index,  eps=1e-3):
    # Extract sc_list and cc_list with numeric keys
    sc_list_numeric, cc_list_numeric = extract_sc_cc_numeric(loaded_results, rep_num, intensity_index)

    y_all = []

    # Iterate over each coalescence pair
    for (c1_idx, c2_idx), cmix in cc_list_numeric.items():
        c1 = sc_list_numeric.get(c1_idx)
        c2 = sc_list_numeric.get(c2_idx)

        if c1 is None or c2 is None:
            print(f"Warning: Sub-community indices {c1_idx}, {c2_idx} not found. Skipping.", file=sys.stderr)
            continue

        # Apply the metric function
        try:
            u, v, k = metric_VectorDecomposition_onlyPositive(c1, c2, cmix)
        except np.linalg.LinAlgError as e:
            if 'Singular matrix' in str(e):
                print("Singular matrix error occurred.")
                print(f"c1: {c1}")
                print(f"c2: {c2}")
                print(f"cmix: {cmix}")
                u=v=1/sqrt(2)
                k=0
        # Calculate y
        x,y=calculate_assymetricity(u,v,k)
        y = characterize_case(x,y)
        y_all.append(y)

    return np.array(y_all)




In [5]:
import numpy as np
import matplotlib.pyplot as plt
from pathos.multiprocessing import ProcessingPool as Pool

# Helper function to evaluate one alpha in parallel
def evaluate_alpha(args):
    alpha, X1_all, X2_all, y_all, order, shuffle_split = args

    mse_train_folds = []
    mse_test_folds = []

    # Cross-validation
    for train_idx, test_idx in shuffle_split.split(X1_all):
        X1_train, X2_train, y_train = X1_all[train_idx], X2_all[train_idx], y_all[train_idx]
        X1_test,  X2_test,  y_test  = X1_all[test_idx],  X2_all[test_idx],  y_all[test_idx]

        # Duplicate X and y
        X1_train = np.vstack([X1_train, -X1_train])
        X2_train = np.vstack([X2_train, -X2_train])
        y_train  = np.hstack([y_train,  -y_train])

        try:
            w_fit = fit_definite_strength_poly(
                X1_train, X2_train, y_train,
                order=order,
                alpha=[0.1, alpha]
            )
        except np.linalg.LinAlgError as e:
            print(f"LinAlgError: {e}")
            continue

        y_pred_test = predict_ratio_poly(w_fit, X1_test, X2_test, order=order)
        y_pred_train = predict_ratio_poly(w_fit, X1_train, X2_train, order=order)

        mse_test_folds.append(np.mean((y_test - y_pred_test)**2) / np.var(y_all))
        mse_train_folds.append(np.mean((y_train - y_pred_train)**2) / np.var(y_all))

    avg_mse_test = np.mean(mse_test_folds)
    avg_mse_train = np.mean(mse_train_folds)

    print(f"Alpha={alpha}: avg test MSE={avg_mse_test:.4f}, avg train MSE={avg_mse_train:.4f}")

    return alpha, avg_mse_test, avg_mse_train


def find_optimal_alpha(
    alpha_list, 
    X1_all, 
    X2_all, 
    y_all, 
    order, 
    shuffle_split, 
    to_plot=False, 
    save_filepath=None
):
    # Parallel evaluation
    pool_args = [(alpha, X1_all, X2_all, y_all, order, shuffle_split) for alpha in alpha_list]

    with Pool(min(len(alpha_list), 10)) as pool:
        results = pool.map(evaluate_alpha, pool_args)

    # Unpack results
    alphas, alpha_mse_test_list, alpha_mse_train_list = zip(*results)

    best_index = np.argmin(alpha_mse_test_list)
    best_alpha = alphas[best_index]
    best_test_mse = alpha_mse_test_list[best_index]

    print(f"\nOptimal alpha: {best_alpha}, Test MSE={best_test_mse:.4f}")

    if to_plot:
        plt.figure(figsize=(6, 4))
        plt.plot(alphas, alpha_mse_test_list, marker='o', label='Test MSE')
        plt.plot(alphas, alpha_mse_train_list, marker='s', label='Train MSE')

        plt.xscale('log')

        plt.plot(best_alpha, best_test_mse, marker='*', markersize=14, 
                color='red', label='Optimal alpha')

        plt.xlabel("Alpha (log scale)")
        plt.ylabel("MSE (normalized)")
        plt.title("Alpha vs. MSE")
        plt.legend()

        if save_filepath:
            plt.savefig(save_filepath, dpi=100, bbox_inches='tight')
            plt.show()
        else:
            plt.show()

    return best_alpha, best_test_mse, alpha_mse_train_list, alpha_mse_test_list


In [12]:
import numpy as np
import json
from pathlib import Path
from sklearn.model_selection import ShuffleSplit

# Your previously defined functions: evaluate_alpha, find_optimal_alpha, fit_definite_strength_poly, predict_ratio_poly

# Define session names and intensity indices
session_names = [
    #"Simulation_Data/new_k_gamma_0_defined_pool_nooverlap_12from48",
    #"Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48",
    #"Simulation_Data/new_k_gamma_0.1_defined_pool_nooverlap_12from48",
    #"Simulation_Data/new_k_gamma_0.15_defined_pool_nooverlap_12from48",
    #"Simulation_Data/new_k_gamma_0.2_defined_pool_nooverlap_12from48",
    #"Simulation_Data/new_k_gamma_0.25_defined_pool_nooverlap_12from48",
    #"Simulation_Data/new_k_gamma_0.5_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48",

]

intensity_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

# Alpha range definition
alpha_list = np.logspace(-4, 0, 30)
rep_nums=8

# Cross-validation setup (adjust parameters as needed)
shuffle_split = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

for session_name in session_names:
    session_results = {}
    for intensity_index in intensity_indices:
        alpha_list_rep = []
        mse_list_rep = []

        for rep_num in range(rep_nums):
            # Load data (customize according to your data structure)
            data_file = Path(session_name) / 'Community.json'
            with open(data_file, 'r') as f:
                loaded_results = json.load(f)

            # Prepare your data (customize this part based on your real data prep functions)
            X1_all, X2_all, y_all = prepare_X1_X2_y(loaded_results, rep_num=rep_num, intensity_index=intensity_index)

            # Run find_optimal_alpha (already parallelized internally)
            best_alpha, best_test_mse, _, _ = find_optimal_alpha(
                alpha_list,
                X1_all,
                X2_all,
                y_all,
                order=1,  # adjust polynomial order as needed
                shuffle_split=shuffle_split,
                to_plot=False
            )

            alpha_list_rep.append(best_alpha)
            mse_list_rep.append(best_test_mse)

            print(f"Rep {rep_num} optimal alpha for {session_name}, intensity {intensity_index}: {best_alpha}, MSE: {best_test_mse}")

        # Store replicate results for this intensity
        session_results[intensity_index] = {
            'best_alphas': alpha_list_rep,
            'best_test_mses': mse_list_rep
        }

    # Save results for the entire session
    output_path = Path(session_name) / "prediction_analysis_optimal_alpha_results.json"

    with open(output_path, 'w') as f:
        json.dump(session_results, f, indent=4)

print("Optimal alpha computation completed and results saved for all sessions and intensities.")

Selected Intensity: 0.0
Selected Community: community_0

Optimal alpha: 0.011721022975334805, Test MSE=0.1813
Rep 0 optimal alpha for Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48, intensity 0: 0.011721022975334805, MSE: 0.18128106736362287
Selected Intensity: 0.0
Selected Community: community_1

Optimal alpha: 0.14873521072935117, Test MSE=0.1187
Rep 1 optimal alpha for Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48, intensity 0: 0.14873521072935117, MSE: 0.11874645824091054
Selected Intensity: 0.0
Selected Community: community_2

Optimal alpha: 0.20433597178569418, Test MSE=0.1251
Rep 2 optimal alpha for Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48, intensity 0: 0.20433597178569418, MSE: 0.12506812345135493
Selected Intensity: 0.0
Selected Community: community_3

Optimal alpha: 0.2807216203941176, Test MSE=0.0767
Rep 3 optimal alpha for Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48, intensity 0: 0.2807216203941176, MSE: 

In [16]:
import json
from pathlib import Path
import numpy as np
from pathos.multiprocessing import ProcessingPool as Pool
from tqdm import tqdm

# Your previously defined functions: prepare_X1_X2_y, prepare_dominance
# session_names and intensity_indices remain unchanged.

# Define your session names and intensity indices
session_names = [
    "Simulation_Data/new_k_gamma_0_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_0.05_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_0.1_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_0.15_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_0.2_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_0.25_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_0.5_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48",
]

intensity_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]

# Wrapper function remains unchanged
def run_task(args):
    session_name, intensity_index, rep_num = args

    output_file = Path(session_name) / 'Community.json'
    with open(output_file, 'r') as f:
        loaded_results = json.load(f)

    x1, x2, y = prepare_X1_X2_y(loaded_results, rep_num, intensity_index)
    dominance = prepare_dominance(loaded_results, rep_num, intensity_index, eps=1e-3)

    return (session_name, intensity_index, rep_num, {
        'dominance': dominance.tolist(),
        'c1': x1.tolist(),
        'c2': x2.tolist(),
        'c3': y.tolist()
    })

tasks = [
    (session_name, intensity_index, rep_num)
    for session_name in session_names
    for intensity_index in intensity_indices
    for rep_num in range(8)
]

aggregated_results = {}

num_workers = min(len(tasks), 16)  # Adjust based on resources
with Pool(num_workers) as pool:
    for session_name, intensity_index, rep_num, result in tqdm(
            pool.uimap(run_task, tasks), total=len(tasks)):

        aggregated_results\
            .setdefault(session_name, {})\
            .setdefault(intensity_index, {})[rep_num] = result

for session_name, session_data in aggregated_results.items():
    result_file = Path(session_name) / 'results_dominance_fractions.json'
    with open(result_file, 'w') as outfile:
        json.dump(session_data, outfile, indent=4)

print("Results saved for each session and intensity index.")


100%|██████████| 768/768 [00:16<00:00, 47.52it/s]


Results saved for each session and intensity index.


In [ ]:

# Define your session names and intensity indices
session_names = [
    "Simulation_Data/standard_defined_pool",
    "Simulation_Data/new_k_gamma_0.5_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_1_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_2_defined_pool_nooverlap_12from48",
    "Simulation_Data/new_k_gamma_4_defined_pool_nooverlap_12from48",
]

intensity_indices = [0, 2, 4, 6, 8, 10]

for session_name in session_names:
    session_results = {}
    for intensity_index in intensity_indices:
        # Initialize the dictionary for each intensity index
        session_results[intensity_index] = {
            'dominance': [],
        }
        for rep_num in np.arange(8):

            # Load the JSON file
            output_file = Path(session_name) / 'Community.json'
            with open(output_file, 'r') as f:
                loaded_results = json.load(f)
                
            # Now running part

            x1,x2,y=prepare_X1_X2_y(loaded_results, rep_num, intensity_index)
            dominance=prepare_dominance(loaded_results, rep_num, intensity_index,  eps=1e-3)
        
            # Store the results for this replicate
            session_results[intensity_index]['dominance']=dominance.tolist()
            session_results[intensity_index]['c1']=x1.tolist()
            session_results[intensity_index]['c2']=x2.tolist()
            session_results[intensity_index]['c3']=y.tolist()

        # Save results for this session and intensity index
        result_file = Path(session_name) / f'results_dominance_fractions.json'
        with open(result_file, 'w') as outfile:
            json.dump(session_results, outfile, indent=4)


print("Results saved for each session and intensity index.")


Selected Intensity: 0.0
Selected Community: community_0
Selected Intensity: 0.0
Selected Community: community_1
Selected Intensity: 0.0
Selected Community: community_2
Selected Intensity: 0.0
Selected Community: community_3
Selected Intensity: 0.0
Selected Community: community_4
Selected Intensity: 0.0
Selected Community: community_5
Selected Intensity: 0.0
Selected Community: community_6
Selected Intensity: 0.0
Selected Community: community_7
Selected Intensity: 0.2
Selected Community: community_0
Selected Intensity: 0.2
Selected Community: community_1
Selected Intensity: 0.2
Selected Community: community_2
Selected Intensity: 0.2
Selected Community: community_3
Selected Intensity: 0.2
Selected Community: community_4
Selected Intensity: 0.2
Selected Community: community_5
Selected Intensity: 0.2
Selected Community: community_6
Selected Intensity: 0.2
Selected Community: community_7
Selected Intensity: 0.4
Selected Community: community_0
Selected Intensity: 0.4
Selected Community: comm

/Users/jysong/Desktop/Gore_lab/Sequencing/Coalescence_session_20230404/Figure_generate/code/common_setup.py:311: RuntimeWarning: divide by zero encountered in true_divide
  y=np.abs(np.abs(np.arctan(np.array(u)/np.array(v)))-np.pi/4)/(np.pi/4)


Selected Intensity: 0.4
Selected Community: community_6
Selected Intensity: 0.4
Selected Community: community_7
Selected Intensity: 0.6000000000000001
Selected Community: community_0
Selected Intensity: 0.6000000000000001
Selected Community: community_1
Selected Intensity: 0.6000000000000001
Selected Community: community_2
Selected Intensity: 0.6000000000000001
Selected Community: community_3
Selected Intensity: 0.6000000000000001
Selected Community: community_4
Selected Intensity: 0.6000000000000001
Selected Community: community_5
Selected Intensity: 0.6000000000000001
Selected Community: community_6
Selected Intensity: 0.6000000000000001
Selected Community: community_7
Selected Intensity: 0.8
Selected Community: community_0
Selected Intensity: 0.8
Selected Community: community_1
Selected Intensity: 0.8
Selected Community: community_2
Selected Intensity: 0.8
Selected Community: community_3
Selected Intensity: 0.8
Selected Community: community_4
Selected Intensity: 0.8
Selected Communi

/Users/jysong/Desktop/Gore_lab/Sequencing/Coalescence_session_20230404/Figure_generate/code/common_setup.py:311: RuntimeWarning: invalid value encountered in true_divide
  y=np.abs(np.abs(np.arctan(np.array(u)/np.array(v)))-np.pi/4)/(np.pi/4)


Selected Intensity: 0.0
Selected Community: community_4
Selected Intensity: 0.0
Selected Community: community_5
Selected Intensity: 0.0
Selected Community: community_6
Selected Intensity: 0.0
Selected Community: community_7
Selected Intensity: 0.2
Selected Community: community_0
Selected Intensity: 0.2
Selected Community: community_1
Selected Intensity: 0.2
Selected Community: community_2
Selected Intensity: 0.2
Selected Community: community_3
Selected Intensity: 0.2
Selected Community: community_4
Selected Intensity: 0.2
Selected Community: community_5
Selected Intensity: 0.2
Selected Community: community_6
Selected Intensity: 0.2
Selected Community: community_7
Selected Intensity: 0.4
Selected Community: community_0
Selected Intensity: 0.4
Selected Community: community_1
Selected Intensity: 0.4
Selected Community: community_2
Selected Intensity: 0.4
Selected Community: community_3
Selected Intensity: 0.4
Selected Community: community_4
Selected Intensity: 0.4
Selected Community: comm

/Users/jysong/Desktop/Gore_lab/Sequencing/Coalescence_session_20230404/Figure_generate/code/common_setup.py:276: RuntimeWarning: invalid value encountered in sqrt
  convert=np.sqrt((1-x3**2)/(x1**2+x2**2))


Selected Intensity: 0.0
Selected Community: community_0
Selected Intensity: 0.0
Selected Community: community_1
Selected Intensity: 0.0
Selected Community: community_2
Selected Intensity: 0.0
Selected Community: community_3
Selected Intensity: 0.0
Selected Community: community_4
Selected Intensity: 0.0
Selected Community: community_5
Selected Intensity: 0.0
Selected Community: community_6
Selected Intensity: 0.0
Selected Community: community_7
Selected Intensity: 0.2
Selected Community: community_0
Selected Intensity: 0.2
Selected Community: community_1
Selected Intensity: 0.2
Selected Community: community_2
Selected Intensity: 0.2
Selected Community: community_3
Selected Intensity: 0.2
Selected Community: community_4
Selected Intensity: 0.2
Selected Community: community_5
Selected Intensity: 0.2
Selected Community: community_6
Selected Intensity: 0.2
Selected Community: community_7
Selected Intensity: 0.4
Selected Community: community_0
Selected Intensity: 0.4
Selected Community: comm